In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

import sqlite3

In [2]:
# Si no está sqlite3
#!pip install sqlite3

# Introducción a SQL en Python con SQLite

Para simplicidad vamos a usar un servidor SQL virtual (no necesitamos usuarios, password, ni ninguna complicación).

In [12]:
# 1) Creamos una base de datos SQLite en memoria
con = sqlite3.connect(":memory:")

In [13]:
# Creamos una tabla de productos
# No estudiamos estos comandos, los usamos para tener una tabla dentro de la base de datos
setup_sql = """
DROP TABLE IF EXISTS productos;

CREATE TABLE productos (
  producto_id INTEGER PRIMARY KEY,
  titulo      TEXT NOT NULL,
  categoria   TEXT NOT NULL,
  precio      REAL NOT NULL
);

INSERT INTO productos(producto_id, titulo, categoria, precio) VALUES
(0,'El Principito', 'libro', 12.00),
(1,'El Declive', 'libro', 18.00),
(2,'go', 'juguete',  25.00),
(3,'ajedrez', 'juguete',  15.00),
(4,'fideos', 'comida',  5.00),
(5,'pan lactal', 'comida',  8.00);

"""

con.executescript(setup_sql)

## SELECT
El comando SELECT nos permite obtener datos de una tabla.

### Sintaxis 1

```
SELECT *
FROM table;
```

Nos da todos los datos de la tabla.

In [14]:
# Leemos toda la tabla

# En Pandas ejecutamos "queries" con read_sql_query(query string, conexión)
productos = pd.read_sql_query("SELECT * FROM productos;", con)
productos

,producto_id,titulo,categoria,precio
0,0,El Principito,libro,12.0
1,1,El Declive,libro,18.0
2,2,go,juguete,25.0
3,3,ajedrez,juguete,15.0
4,4,fideos,comida,5.0
5,5,pan lactal,comida,8.0


**Observación:** En general los queries en SQL no distinguen mayusculas de mínusculas, pero suele ser común poner los comandos en mayúscula para distinguirlos fácilmente de nombres de variables.

### Sintaxis 2

```
SELECT columna1, columna2, ...
FROM table;
```
Nos da solo las las columnas indicadas (en situaciones reales, esto es más común que usar *, para no traer columnas innecesarias).

In [16]:
# Leemos las columnas titulo y precio
productos = pd.read_sql_query("SELECT titulo, precio FROM productos;", con)
productos

,titulo,precio
0,El Principito,12.0
1,El Declive,18.0
2,go,25.0
3,ajedrez,15.0
4,fideos,5.0
5,pan lactal,8.0


In [17]:
# OJO! Esto da error.
# En SQL normalmente no usamos paréntesis para listar columnas (ni hace falta comillas para los nombres de columnas o tablas)
productos = pd.read_sql_query("SELECT (titulo, precio) FROM productos;", con)
productos

DatabaseError: Execution failed on sql 'SELECT (titulo, precio) FROM productos;': row value misused

### Sintaxis 3

```
SELECT columna1, columna2, ...
FROM table
WHERE condiciones;
```
Nos da solo las columnas indicadas de las filas que cumplan las condiciones.

In [18]:
# Leemos las columnas titulo y precio de articulos que cuestan mas de 10 pesos
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE precio > 10;", con)
productos

,titulo,precio
0,El Principito,12.0
1,El Declive,18.0
2,go,25.0
3,ajedrez,15.0


In [9]:
# Si queremos los articulos con precio entre 10 y 20...
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE precio > 10 AND precio < 20;", con)
productos

,titulo,precio
0,El Principito,12.0
1,El Declive,18.0
2,ajedrez,15.0


In [19]:
# Acá sí podemos poner paréntesis si la condición es más compleja
# Usamos comillas simples para strings
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE (precio > 10) AND (categoria = 'juguete' OR categoria = 'libro');", con)
productos

,titulo,precio
0,El Principito,12.0
1,El Declive,18.0
2,go,25.0
3,ajedrez,15.0


## Ejercicio
Rehacemos algunos filtros de clases anteriores.
1. De la base Gapminder, seleccionar todas las filas del año 2007.
2. De la base Gapminder, seleccionar todas las filas de Argentina.
3. De la base MPG seleccionar las columnas mpg, horsepower y model_year de todos los modelos de los años 1970, 1975 y 1980.

Sugerencia para el último ítem
```
SELECT column1, column2, ...
FROM table_name
WHERE column_name IN (value1, value2, ...);
```

In [20]:
# Cargamos los DataFrames
from gapminder import gapminder
mpg = sns.load_dataset("mpg")

In [21]:
# Para convertir un DataFrame a SQL:
conGap = sqlite3.connect(":memory:")   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.

# Guardar el DataFrame como tabla "gapminder"
gapminder.to_sql("gapminder", conGap, index=False, if_exists="replace")

1704

(devolvió la cantidad de filas afectadas)

In [22]:
# Verificamos (primeras filas)
pd.read_sql_query("SELECT * FROM gapminder LIMIT 5;", conGap)

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


**Importante:** Esta forma de trabajar (primero cargar el DataFrame, pasarlo a SQL y despues volver a convertirlo a DataFrame) es claramente muy ineficiente. La utilizamos solo con fines **didácticos** para no montar un servidor SQL.

Tenemos que pensar que en el uso real, los datos ya van a estar en un servidor SQL.

In [ ]:
# Seleccionamos los datos del año 2007


In [ ]:
# Seleccionamos los datos de Argentina


In [ ]:
# columnas mpg, horsepower y model_year de todos los modelos de los años 1970, 1975 y 1980
conMPG = ???   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.


## Agregamos datos con GROUP BY
```
SELECT columnas, funcion_de_agregacion(columna)
FROM tabla
WHERE condicion
GROUP BY columna_agrupacion;
```

**Ejemplo.** Calcular la población total por continente en 2007 y graficar.

In [24]:
# Triple comillas para texto multi-linea, es común para comandos SQL poner cada parte en una línea distinta,
# pero es una sola instrucción (no estamos encadenando métodos como hacemos en Pandas)!
pd.read_sql_query("""    
SELECT
    continent, year, SUM(pop)
FROM gapminder
WHERE year = 2007
GROUP BY continent
""", conGap)

,continent,year,SUM(pop)
0,Africa,2007,929539692
1,Americas,2007,898871184
2,Asia,2007,3811953827
3,Europe,2007,586098529
4,Oceania,2007,24549947


### Para renombrar la columna
```
SELECT columnas_agrupacion, funcion_de_agregacion(columna) as nombre_columna
FROM tabla
WHERE condicion
GROUP BY columna_agrupacion;
```

In [25]:
pd.read_sql_query("""    
SELECT
    continent, year, SUM(pop) as poblacion_total
FROM gapminder
WHERE year = 2007
GROUP BY continent
""", conGap)

,continent,year,poblacion_total
0,Africa,2007,929539692
1,Americas,2007,898871184
2,Asia,2007,3811953827
3,Europe,2007,586098529
4,Oceania,2007,24549947


In [34]:
# Y si queremos la poblacion total mundial en cada año?
# Funciona esto?
pob = pd.read_sql_query("""    
SELECT
    continent,
    year,
    SUM(pop) as poblacion_total
FROM gapminder
GROUP BY continent
""", conGap)

In [35]:
pob

,continent,year,poblacion_total
0,Africa,1952,6187585961
1,Americas,1952,7351438499
2,Asia,1952,30507333901
3,Europe,1952,6181115304
4,Oceania,1952,212992136


In [ ]:
# Cómo lo arreglamos?


### Ejercicio
Utilizando SQL generar una tabla con el total de pasajeros por año de la tabla de pasajes de avión y graficar.

In [33]:
flights = sns.load_dataset("flights")
flights

,year,month,passengers
0,1949,Jan,112
1,1949,Feb,118
2,1949,Mar,132
3,1949,Apr,129
4,1949,May,121
...,...,...,...
139,1960,Aug,606
140,1960,Sep,508
141,1960,Oct,461
142,1960,Nov,390


### Ejercicio
1. Utilizando SQL generar una tabla con el peso promedio de los pingüinos por especie. Función: AVG(columna)
2. Utilizando SQL generar una tabla con el peso promedio de los pingüinos por especie e isla (es decir un valor para cada combinación de isla y especie).
3. Utilizando SQL generar una tabla con la cantidad de  pingüinos por especie e isla (es decir un valor para cada combinación de isla y especie en la que haya al menos un pingüino). Función: COUNT(*) (no contamos una columna en particular, sino filas)

In [ ]:
penguins = sns.load_dataset("penguins")
penguins

# Más adelante...
Vamos a ver cómo combinar información en distintas tablas, donde vamos a ver mejor el potencial de SQL.